# Phase 2: Comprehensive Evaluation of Semantic Region Selection

## Goals
1. **Full training**: Use the complete HotpotQA train set (~90K) to eliminate data scarcity issues
2. **Multi-dataset**: Evaluate on HotpotQA + SQuAD + Natural Questions (via constructed retrieval)
3. **Best-of-each-form**: Each geometric form gets its own hyperparameter sweep to show peak performance
4. **Practical deployment analysis**: Latency, memory, scalability, cost-benefit
5. **Publication-quality figures**: All English, no Chinese characters, with Oracle + Cosine baselines

## Geometric Forms Compared
| ID | Form | Shape | Key property |
|---|---|---|---|
| B1 | Oracle | Perfect | Upper bound |
| B2 | Random | None | Lower bound |
| B3 | Cosine Top-K | Sphere (point) | Equal-weight dimensions |
| M1 | Diagonal Gaussian | Axis-aligned ellipsoid | Per-dim weighting |
| M2 | Low-Rank Gaussian | Rotated ellipsoid | Dim interaction |
| M3 | Skew-Gaussian | Skewed ellipsoid | Directional asymmetry |
| M4 | Mix-Skew K=2 | 2 skewed ellipsoids | Multi-center |
| M5 | Mix-Skew K=3 | 3 skewed ellipsoids | More flexible |

In [ ]:
%%capture
!pip install transformers datasets sentence-transformers accelerate
!pip install bert-score rouge-score scipy scikit-learn matplotlib tqdm

In [ ]:
import os, json, random, time, math, gc, numpy as np, torch, torch.nn as nn, torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
import matplotlib
matplotlib.use('Agg')  # non-interactive backend
import matplotlib.pyplot as plt
from collections import defaultdict, Counter, OrderedDict
from sklearn.metrics.pairwise import cosine_similarity
from scipy import stats

# ====== Ensure English-only rendering (no CJK issues) ======
plt.rcParams.update({
    'figure.dpi': 150, 'savefig.dpi': 300, 'savefig.bbox': 'tight',
    'font.family': 'serif',
    'font.serif': ['DejaVu Serif', 'Times New Roman', 'serif'],
    'mathtext.fontset': 'dejavuserif',
    'axes.unicode_minus': False,
    'axes.labelsize': 10, 'axes.titlesize': 11,
    'xtick.labelsize': 8, 'ytick.labelsize': 8, 'legend.fontsize': 8,
    'axes.linewidth': 0.8, 'axes.grid': True, 'grid.alpha': 0.3,
})

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}, VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB')

---
## Part 1: Data Preparation (Full Scale)

In [ ]:
from datasets import load_dataset
from sentence_transformers import SentenceTransformer

# ====== Dataset 1: HotpotQA (FULL train set) ======
print('Loading HotpotQA...')
hp_train_raw = load_dataset('hotpot_qa', 'distractor', split='train')
hp_val_raw = load_dataset('hotpot_qa', 'distractor', split='validation')

def parse_hp(s):
    sup = set(s['supporting_facts']['title'])
    return {'query': s['question'], 'answer': s['answer'], 'type': s.get('type',''),
            'contexts': [{'title': t, 'text': ' '.join(sn), 'is_supporting': t in sup}
                         for t, sn in zip(s['context']['title'], s['context']['sentences'])]}

hp_train_all = [parse_hp(s) for s in hp_train_raw]
hp_val_all = [parse_hp(s) for s in hp_val_raw]
random.shuffle(hp_train_all)
random.shuffle(hp_val_all)

# Use up to 20K train (balancing full-scale with T4 time constraints)
# If on A100/H100, increase to 50K or full 90K
MAX_TRAIN = 20000
hp_train = hp_train_all[:MAX_TRAIN]
hp_val = hp_val_all[:1000]
hp_test = hp_val_all[1000:3000]
print(f'  HotpotQA: train={len(hp_train)}, val={len(hp_val)}, test={len(hp_test)}')
print(f'  Test type distribution: {Counter(d["type"] for d in hp_test)}')

# ====== Dataset 2: SQuAD 2.0 (single-hop factoid) ======
print('Loading SQuAD 2.0...')
squad_raw = load_dataset('squad_v2', split='validation')
has_answer = [s for s in squad_raw if s['answers']['text']]
all_ctxs = list(set(s['context'] for s in has_answer))
print(f'  SQuAD: {len(has_answer)} answerable, {len(all_ctxs)} unique paragraphs')

def build_squad(samples, pool, n_neg=9):
    out = []
    for s in samples:
        pos = s['context']
        negs = [c for c in pool if c != pos]
        if len(negs) < n_neg: continue
        ctxs = [{'text': pos, 'is_supporting': True}] + \
               [{'text': c, 'is_supporting': False} for c in random.sample(negs, n_neg)]
        random.shuffle(ctxs)
        out.append({'query': s['question'], 'answer': s['answers']['text'][0],
                    'type': 'factoid', 'contexts': ctxs})
    return out

sq_all = build_squad(has_answer, all_ctxs)
random.shuffle(sq_all)
sq_train = sq_all[:3000]
sq_val = sq_all[3000:3500]
sq_test = sq_all[3500:5000]
print(f'  SQuAD retrieval: train={len(sq_train)}, val={len(sq_val)}, test={len(sq_test)}')

datasets = {
    'HotpotQA': {'train': hp_train, 'val': hp_val, 'test': hp_test, 'k': 2},
    'SQuAD':    {'train': sq_train, 'val': sq_val, 'test': sq_test, 'k': 1},
}

In [ ]:
# ====== Encode with E5-large ======
print('Loading E5-large-v2...')
enc = SentenceTransformer('intfloat/e5-large-v2', device=DEVICE)
DIM = enc.get_sentence_embedding_dimension()
print(f'Embedding dim: {DIM}')

def encode_dataset(data, encoder, bs=64):
    qs = [d['query'] for d in data]
    ct, cm = [], []
    for i, d in enumerate(data):
        for j, c in enumerate(d['contexts']):
            ct.append(c['text'][:512]); cm.append((i,j))
    print(f'  Encoding {len(qs)} queries + {len(ct)} contexts...')
    qe = encoder.encode(qs, batch_size=bs, show_progress_bar=True, convert_to_numpy=True)
    ce = encoder.encode(ct, batch_size=bs, show_progress_bar=True, convert_to_numpy=True)
    result = []
    for i, d in enumerate(data):
        s = dict(d); s['query_emb'] = qe[i]; s['context_embs'] = []; result.append(s)
    for (i,j), emb in zip(cm, ce): result[i]['context_embs'].append(emb)
    for s in result: s['context_embs'] = np.array(s['context_embs'])
    return result

encoded = {}
for name, ds in datasets.items():
    print(f'\nEncoding {name}...')
    encoded[name] = {}
    for split in ['train', 'val', 'test']:
        print(f'  {split}:')
        encoded[name][split] = encode_dataset(ds[split], enc)

del enc; torch.cuda.empty_cache(); gc.collect()
print('\nAll encoding complete.')

---
## Part 2: Model Definitions (All Geometric Forms)

In [ ]:
class Backbone(nn.Module):
    def __init__(self, d, h=256):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d,h), nn.GELU(), nn.Dropout(0.1),
                                 nn.Linear(h,h), nn.GELU(), nn.Dropout(0.1))
    def forward(self, q): return self.net(q)

# ====== M1: Diagonal Gaussian ======
class DiagonalGaussian(nn.Module):
    def __init__(self, d, h=256, clamp=(-3,3)):
        super().__init__()
        self.cmin, self.cmax = clamp
        self.backbone = Backbone(d, h)
        self.head = nn.Linear(h, d)
        nn.init.zeros_(self.head.weight); nn.init.zeros_(self.head.bias)
    def forward(self, q): return torch.clamp(self.backbone(q), self.cmin, self.cmax)
    def score(self, q, ctx):
        lv = self.head(self.backbone(q))
        lv = torch.clamp(lv, self.cmin, self.cmax)
        mu = q.unsqueeze(1); diff = ctx - mu
        var = torch.exp(lv.unsqueeze(1))
        return -0.5 * (torch.sum(diff**2/var, dim=-1) + torch.sum(lv, dim=-1, keepdim=True))

# ====== M2: Low-Rank Symmetric Gaussian ======
class LowRankGaussian(nn.Module):
    def __init__(self, d, rank=16, h=256, clamp=(-3,3)):
        super().__init__()
        self.d, self.rank, self.cmin, self.cmax = d, rank, clamp[0], clamp[1]
        self.backbone = Backbone(d, h)
        self.head_d = nn.Linear(h, d)
        self.head_L = nn.Linear(h, d*rank)
        nn.init.zeros_(self.head_d.weight); nn.init.zeros_(self.head_d.bias)
        nn.init.normal_(self.head_L.weight, std=0.01); nn.init.zeros_(self.head_L.bias)
    def score(self, q, ctx):
        h = self.backbone(q); B,N,D = ctx.shape; R = self.rank
        log_d = torch.clamp(self.head_d(h), self.cmin, self.cmax)
        L = self.head_L(h).view(B, D, R) * 0.1
        d_val = torch.exp(log_d); d_inv = 1.0/d_val
        diff = ctx - q.unsqueeze(1)
        mahal = torch.sum(diff**2 * d_inv.unsqueeze(1), dim=-1)
        DinvL = d_inv.unsqueeze(-1) * L
        M = torch.eye(R, device=q.device).unsqueeze(0) + torch.bmm(L.transpose(1,2), DinvL)
        Mi = torch.linalg.inv(M)
        v = diff * d_inv.unsqueeze(1)
        w = torch.bmm(L.transpose(1,2), v.transpose(1,2))
        mahal = mahal - torch.sum(w * torch.bmm(Mi, w), dim=1)
        ld = torch.sum(log_d, -1, keepdim=True) + torch.linalg.slogdet(M)[1].unsqueeze(-1)
        return -0.5 * (mahal + ld)

# ====== M3: Skew-Gaussian ======
class SkewGaussian(nn.Module):
    def __init__(self, d, rank=16, h=256, clamp=(-3,3)):
        super().__init__()
        self.d, self.rank, self.cmin, self.cmax = d, rank, clamp[0], clamp[1]
        self.backbone = Backbone(d, h)
        self.head_d = nn.Linear(h, d)
        self.head_L = nn.Linear(h, d*rank)
        self.head_delta = nn.Linear(h, d)
        self.head_alpha = nn.Linear(h, d)
        for hd in [self.head_d, self.head_delta, self.head_alpha]:
            nn.init.zeros_(hd.weight); nn.init.zeros_(hd.bias)
        nn.init.normal_(self.head_L.weight, std=0.01); nn.init.zeros_(self.head_L.bias)
    def score(self, q, ctx):
        h = self.backbone(q); B,N,D = ctx.shape; R = self.rank
        log_d = torch.clamp(self.head_d(h), self.cmin, self.cmax)
        L = self.head_L(h).view(B, D, R) * 0.1
        delta = self.head_delta(h) * 0.1
        alpha = self.head_alpha(h)
        mu = q + delta; diff = ctx - mu.unsqueeze(1)
        d_val = torch.exp(log_d); d_inv = 1.0/d_val
        mahal = torch.sum(diff**2 * d_inv.unsqueeze(1), dim=-1)
        DinvL = d_inv.unsqueeze(-1) * L
        M = torch.eye(R, device=q.device).unsqueeze(0) + torch.bmm(L.transpose(1,2), DinvL)
        Mi = torch.linalg.inv(M)
        v = diff * d_inv.unsqueeze(1)
        w = torch.bmm(L.transpose(1,2), v.transpose(1,2))
        mahal = mahal - torch.sum(w * torch.bmm(Mi, w), dim=1)
        ld = torch.sum(log_d, -1, keepdim=True) + torch.linalg.slogdet(M)[1].unsqueeze(-1)
        sym = -0.5 * (mahal + ld)
        skew = F.logsigmoid(torch.sum(alpha.unsqueeze(1) * diff, dim=-1))
        return sym + skew

# ====== M4/M5: Mixture of Skew-Gaussians ======
class MixSkewGaussian(nn.Module):
    def __init__(self, d, K=2, rank=16, h=256, clamp=(-3,3)):
        super().__init__()
        self.d, self.K, self.rank = d, K, rank
        self.cmin, self.cmax = clamp
        self.backbone = Backbone(d, h)
        self.head_L = nn.Linear(h, d*rank)
        self.head_pi = nn.Linear(h, K)
        self.head_d = nn.Linear(h, K*d)
        self.head_delta = nn.Linear(h, K*d)
        self.head_alpha = nn.Linear(h, K*d)
        for hd in [self.head_pi, self.head_d, self.head_delta, self.head_alpha]:
            nn.init.zeros_(hd.weight); nn.init.zeros_(hd.bias)
        nn.init.normal_(self.head_L.weight, std=0.01); nn.init.zeros_(self.head_L.bias)
    def score(self, q, ctx):
        h = self.backbone(q); B,N,D = ctx.shape; R = self.rank; K = self.K
        lpi = F.log_softmax(self.head_pi(h), dim=-1)
        L = self.head_L(h).view(B, D, R) * 0.1
        all_ld = torch.clamp(self.head_d(h).view(B,K,D), self.cmin, self.cmax)
        all_delta = self.head_delta(h).view(B,K,D) * 0.1
        all_alpha = self.head_alpha(h).view(B,K,D)
        comp = []
        for k in range(K):
            mu_k = q + all_delta[:,k,:]; diff = ctx - mu_k.unsqueeze(1)
            log_d = all_ld[:,k,:]; alpha = all_alpha[:,k,:]
            d_val = torch.exp(log_d); d_inv = 1.0/d_val
            mahal = torch.sum(diff**2 * d_inv.unsqueeze(1), dim=-1)
            DinvL = d_inv.unsqueeze(-1) * L
            Mk = torch.eye(R, device=q.device).unsqueeze(0) + torch.bmm(L.transpose(1,2), DinvL)
            Mi = torch.linalg.inv(Mk)
            v = diff * d_inv.unsqueeze(1)
            w = torch.bmm(L.transpose(1,2), v.transpose(1,2))
            mahal = mahal - torch.sum(w * torch.bmm(Mi, w), dim=1)
            ld = torch.sum(log_d, -1, keepdim=True) + torch.linalg.slogdet(Mk)[1].unsqueeze(-1)
            sym = -0.5 * (mahal + ld)
            skew = F.logsigmoid(torch.sum(alpha.unsqueeze(1) * diff, dim=-1))
            comp.append(lpi[:,k:k+1] + sym + skew)
        return torch.logsumexp(torch.stack(comp, 0), dim=0)

# Print parameter counts
print('Model parameter counts:')
for name, cls, kw in [
    ('M1: Diagonal', DiagonalGaussian, {}),
    ('M2: LowRank (r=16)', LowRankGaussian, {'rank': 16}),
    ('M3: Skew (r=16)', SkewGaussian, {'rank': 16}),
    ('M4: MixSkew K=2', MixSkewGaussian, {'K': 2, 'rank': 16}),
    ('M5: MixSkew K=3', MixSkewGaussian, {'K': 3, 'rank': 16}),
]:
    m = cls(DIM, **kw)
    n = sum(p.numel() for p in m.parameters())
    print(f'  {name:25s}: {n:>10,} ({n/1e6:.1f}M)')

In [ ]:
import os, json, random, time, math, gc, numpy as np, torch, torch.nn as nn, torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
import matplotlib
matplotlib.use('Agg')  # non-interactive backend
import matplotlib.pyplot as plt
from collections import defaultdict, Counter, OrderedDict
from sklearn.metrics.pairwise import cosine_similarity
from scipy import stats

# ====== Ensure English-only rendering (no CJK issues) ======
plt.rcParams.update({
    'figure.dpi': 150, 'savefig.dpi': 300, 'savefig.bbox': 'tight',
    'font.family': 'serif',
    'font.serif': ['DejaVu Serif', 'Times New Roman', 'serif'],
    'mathtext.fontset': 'dejavuserif',
    'axes.unicode_minus': False,
    'axes.labelsize': 10, 'axes.titlesize': 11,
    'xtick.labelsize': 8, 'ytick.labelsize': 8, 'legend.fontsize': 8,
    'axes.linewidth': 0.8, 'axes.grid': True, 'grid.alpha': 0.3,
})

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}, VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB')

# ====== Training infrastructure ======

class CtxDS(Dataset):
    def __init__(self, data): self.data = data
    def __len__(self): return len(self.data)
    def __getitem__(self, i):
        s = self.data[i]
        return (torch.tensor(s['query_emb'], dtype=torch.float32),
                torch.tensor(s['context_embs'], dtype=torch.float32),
                torch.tensor([1.0 if c['is_supporting'] else 0.0 for c in s['contexts']], dtype=torch.float32))

def custom_collate_fn(batch):
    queries = torch.stack([item[0] for item in batch])
    context_embs_list = [item[1] for item in batch]
    labels_list = [item[2] for item in batch]

    # Pad context_embs and labels to the max number of contexts in the batch
    max_num_contexts = max(ce.shape[0] for ce in context_embs_list)
    padded_context_embs = []
    padded_labels = []
    for ce, lbs in zip(context_embs_list, labels_list):
        num_contexts = ce.shape[0]
        padding_needed = max_num_contexts - num_contexts
        # Pad context_embs with zeros
        padded_ce = F.pad(ce, (0, 0, 0, padding_needed), 'constant', 0.0)
        padded_context_embs.append(padded_ce)
        # Pad labels with zeros (or a value indicating no context)
        padded_lb = F.pad(lbs, (0, padding_needed), 'constant', 0.0)
        padded_labels.append(padded_lb)

    padded_context_embs = torch.stack(padded_context_embs)
    padded_labels = torch.stack(padded_labels)

    return queries, padded_context_embs, padded_labels

def infonce(scores, labels, temp=0.02):
    B, N = scores.shape; pm, nm = labels.bool(), ~labels.bool()
    loss = torch.tensor(0.0, device=scores.device); n = 0
    for b in range(B):
        ps, ns = scores[b][pm[b]], scores[b][nm[b]]
        # Filter out padded contexts for loss calculation
        if len(ps)==0 or len(ns)==0: continue
        for p in ps:
            logits = torch.cat([p.unsqueeze(0), ns]) / temp
            loss += F.cross_entropy(logits.unsqueeze(0), torch.tensor([0], device=logits.device)) # Target is 0, the positive sample
            n += 1
    return loss / max(n,1)

def sel_cos(q, c, k=2):
    s = cosine_similarity(q.reshape(1,-1), c)[0]
    return np.argsort(s)[-k:][::-1], s

def make_sel(model):
    def _s(q, c, k=2):
        model.eval()
        with torch.no_grad():
            qt = torch.tensor(q, dtype=torch.float32).unsqueeze(0).to(DEVICE)
            # Ensure contexts are properly shaped for model.score (B, N, D)
            # If c is already (N, D), unsqueeze to (1, N, D)
            if len(c.shape) == 2:
                ct = torch.tensor(c, dtype=torch.float32).unsqueeze(0).to(DEVICE)
            else:
                ct = torch.tensor(c, dtype=torch.float32).to(DEVICE)
            sc = model.score(qt, ct)[0].cpu().numpy()
        return np.argsort(sc)[-k:][::-1], sc
    return _s

def eval_f1(data, sel_fn, k=2):
    f1s = []
    for s in data:
        sel, _ = sel_fn(s['query_emb'], s['context_embs'], k=k)
        labs = [1.0 if c['is_supporting'] else 0.0 for c in s['contexts']]
        hit = sum(1 for i in sel if labs[i] == 1.0); p = hit/k; r = hit/max(sum(labs),1)
        f1s.append(2*p*r/(p+r) if (p+r)>0 else 0)
    return np.array(f1s)

def train_model(model, train_data, val_data, k_eval=2, epochs=30, lr=5e-4, temp=0.02, patience=5):
    """Train with early stopping based on validation F1."""
    loader = DataLoader(CtxDS(train_data), batch_size=32, shuffle=True, num_workers=0, collate_fn=custom_collate_fn)
    opt = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    best_f1, best_st, no_improve = -1, None, 0
    hist = defaultdict(list)
    for ep in range(epochs):
        model.train(); el, nb = 0, 0
        for q, c, l in loader:
            q,c,l = q.to(DEVICE), c.to(DEVICE), l.to(DEVICE)
            loss = infonce(model.score(q, c), l, temp)
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); el += loss.item(); nb += 1
        sched.step()
        vf = eval_f1(val_data, make_sel(model), k=k_eval).mean()
        hist['loss'].append(el/nb); hist['val_f1'].append(vf)
        if vf > best_f1:
            best_f1 = vf; best_st = {k:v.clone() for k,v in model.state_dict().items()}; no_improve = 0
        else:
            no_improve += 1
        if (ep+1) % 10 == 0 or ep == 0 or no_improve == 0:
            print(f'  Epoch {ep+1:3d}/{epochs} | loss={el/nb:.4f} | val_f1={vf:.4f} | best={best_f1:.4f}')
        if no_improve >= patience:
            print(f'  Early stopping at epoch {ep+1} (no improvement for {patience} epochs)')
            break
    model.load_state_dict(best_st)
    return model, dict(hist), best_f1

print('Training infrastructure ready.')

---
## Part 3: Train All Forms on HotpotQA (Full Data)

Each form gets its best hyperparameters from Phase 1 ablations.

In [ ]:
# ====== Train all geometric forms on HotpotQA ======
hp_tr = encoded['HotpotQA']['train']
hp_va = encoded['HotpotQA']['val']
hp_te = encoded['HotpotQA']['test']

RANK = 16
TEMP = 0.02
EPOCHS = 40  # more epochs since more data
PATIENCE = 8

hp_models = OrderedDict()

print('='*60)
print('M1: Diagonal Gaussian')
m1 = DiagonalGaussian(DIM).to(DEVICE)
m1, h1, f1_1 = train_model(m1, hp_tr, hp_va, k_eval=2, epochs=EPOCHS, temp=TEMP, patience=PATIENCE)
hp_models['Diagonal'] = {'model': m1, 'hist': h1}

print('\n' + '='*60)
print('M2: Low-Rank Gaussian (r=16)')
m2 = LowRankGaussian(DIM, rank=RANK).to(DEVICE)
m2, h2, f1_2 = train_model(m2, hp_tr, hp_va, k_eval=2, epochs=EPOCHS, temp=TEMP, patience=PATIENCE)
hp_models['LowRank'] = {'model': m2, 'hist': h2}

print('\n' + '='*60)
print('M3: Skew-Gaussian (r=16)')
m3 = SkewGaussian(DIM, rank=RANK).to(DEVICE)
m3, h3, f1_3 = train_model(m3, hp_tr, hp_va, k_eval=2, epochs=EPOCHS, temp=TEMP, patience=PATIENCE)
hp_models['Skew'] = {'model': m3, 'hist': h3}

print('\n' + '='*60)
print('M4: Mixture Skew K=2 (r=16)')
m4 = MixSkewGaussian(DIM, K=2, rank=RANK).to(DEVICE)
m4, h4, f1_4 = train_model(m4, hp_tr, hp_va, k_eval=2, epochs=EPOCHS, temp=TEMP, patience=PATIENCE)
hp_models['MixSkew-2'] = {'model': m4, 'hist': h4}

print('\n' + '='*60)
print('M5: Mixture Skew K=3 (r=16)')
m5 = MixSkewGaussian(DIM, K=3, rank=RANK).to(DEVICE)
m5, h5, f1_5 = train_model(m5, hp_tr, hp_va, k_eval=2, epochs=EPOCHS, temp=TEMP, patience=PATIENCE)
hp_models['MixSkew-3'] = {'model': m5, 'hist': h5}

print('\nAll HotpotQA models trained.')

---
## Part 4: Comprehensive Evaluation on HotpotQA

In [ ]:
# ====== Full evaluation on HotpotQA test set ======

results_hp = OrderedDict()

# Oracle
oracle_f1 = np.array([1.0 if sum(c['is_supporting'] for c in s['contexts']) >= 2 else
    sum(c['is_supporting'] for c in s['contexts'])/2.0 for s in hp_te])
results_hp['Oracle'] = {'f1': oracle_f1, 'color': '#238B45'}

# Random
results_hp['Random'] = {'f1': eval_f1(hp_te, lambda q,c,k=2: (np.random.choice(len(c),k,replace=False), None), k=2), 'color': '#BDBDBD'}

# Cosine
results_hp['Cosine'] = {'f1': eval_f1(hp_te, sel_cos, k=2), 'color': '#969696'}

# All models
model_colors = {'Diagonal': '#6BAED6', 'LowRank': '#4292C6', 'Skew': '#FD8D3C',
                'MixSkew-2': '#CB181D', 'MixSkew-3': '#8C2D04'}
for mname, mdata in hp_models.items():
    results_hp[mname] = {'f1': eval_f1(hp_te, make_sel(mdata['model']), k=2), 'color': model_colors[mname]}

# Print summary
cos_m = results_hp['Cosine']['f1'].mean()
print('='*85)
print(f'HotpotQA Test Results (n={len(hp_te)}, k=2, train={len(hp_tr)})')
print('='*85)
print(f'{"Method":20s} | {"F1":>8s} | {"vs Cosine":>10s} | {"p-value":>12s}')
print('-'*60)
for rn, rd in results_hp.items():
    fm = rd['f1'].mean(); d = fm - cos_m
    if rn in ('Oracle','Random','Cosine'):
        print(f'{rn:20s} | {fm:8.4f} | {d:+10.4f} | {"---":>12s}')
    else:
        _, pv = stats.ttest_rel(rd['f1'], results_hp['Cosine']['f1'])
        sig = '***' if pv<0.001 else '**' if pv<0.01 else '*' if pv<0.05 else 'ns'
        mk = '*' if d > 0 else ' '
        print(f'{mk}{rn:19s} | {fm:8.4f} | {d:+10.4f} | {pv:.6f} {sig}')
print('='*85)

In [ ]:
# ====== Compression rate analysis (k=1..5) ======
comp_hp = {}
for rn in ['Cosine'] + list(hp_models.keys()):
    sf = sel_cos if rn == 'Cosine' else make_sel(hp_models[rn]['model'])
    comp_hp[rn] = {k: eval_f1(hp_te, sf, k=k) for k in [1,2,3,4,5]}

print('\nCompression rate analysis (HotpotQA):')
print(f'{"Method":15s}', end='')
for k in [1,2,3,4,5]: print(f' | k={k:d}    ', end='')
print()
for rn, cd in comp_hp.items():
    print(f'{rn:15s}', end='')
    for k in [1,2,3,4,5]: print(f' | {cd[k].mean():.4f}', end='')
    print()

In [ ]:
# ====== Question type analysis ======
types = sorted(set(d['type'] for d in hp_test))
type_idx = {t: [i for i,d in enumerate(hp_test) if d['type']==t] for t in types}

print('\nF1 by question type (k=2):')
print(f'{"Method":15s}', end='')
for t in types: print(f' | {t:>12s}', end='')
print()
for rn in ['Cosine'] + list(hp_models.keys()):
    f1arr = results_hp[rn]['f1']
    print(f'{rn:15s}', end='')
    for t in types:
        tf1 = np.array([f1arr[i] for i in type_idx[t]]).mean()
        print(f' | {tf1:12.4f}', end='')
    print()

---
## Part 5: Cross-Dataset Evaluation (SQuAD)

In [ ]:
# ====== Train best model (Skew) on SQuAD and evaluate ======
sq_tr = encoded['SQuAD']['train']
sq_va = encoded['SQuAD']['val']
sq_te = encoded['SQuAD']['test']

print('Training Skew-Gaussian on SQuAD...')
sq_model = SkewGaussian(DIM, rank=RANK).to(DEVICE)
sq_model, sq_hist, sq_best = train_model(sq_model, sq_tr, sq_va, k_eval=1, epochs=30, temp=TEMP, patience=8)

sq_cos = eval_f1(sq_te, sel_cos, k=1)
sq_our = eval_f1(sq_te, make_sel(sq_model), k=1)
_, sq_pv = stats.ttest_rel(sq_our, sq_cos)
print(f'\nSQuAD results: Cosine={sq_cos.mean():.4f}, Skew={sq_our.mean():.4f}, '
      f'delta={sq_our.mean()-sq_cos.mean():+.4f}, p={sq_pv:.6f}')

---
## Part 6: Generation Quality

In [ ]:
# ====== Fix bert_score compatibility with newer transformers ======
!pip install -q bert-score==0.3.13

from transformers import T5ForConditionalGeneration, T5Tokenizer
from rouge_score import rouge_scorer

print('Loading Flan-T5-base...')
gen_tok = T5Tokenizer.from_pretrained('google/flan-t5-base')
gen_mod = T5ForConditionalGeneration.from_pretrained('google/flan-t5-base').to(DEVICE)
gen_mod.eval()

def gen_answer(q, ctxs):
    prompt = f"Answer the question based on the context.\n\nContext: {' '.join(ctxs)}\n\nQuestion: {q}\n\nAnswer:"
    inp = gen_tok(prompt, return_tensors='pt', max_length=512, truncation=True).to(DEVICE)
    with torch.no_grad():
        out = gen_mod.generate(**inp, max_new_tokens=64, num_beams=2, early_stopping=True)
    return gen_tok.decode(out[0], skip_special_tokens=True)

def compute_bertscore(preds, refs):
    """Compute BERTScore with fallback for compatibility issues."""
    try:
        from bert_score import score as bert_score_fn
        P, R, F1 = bert_score_fn(preds, refs, lang='en', verbose=False)
        return F1.mean().item()
    except (AttributeError, Exception) as e:
        print(f'  BERTScore unavailable ({type(e).__name__}), using fallback sentence-similarity...')
        # Fallback: use sentence-transformers cosine similarity as proxy
        from sentence_transformers import SentenceTransformer
        st = SentenceTransformer('all-MiniLM-L6-v2', device=DEVICE)
        p_emb = st.encode(preds, show_progress_bar=False)
        r_emb = st.encode(refs, show_progress_bar=False)
        sims = [cosine_similarity(p.reshape(1,-1), r.reshape(1,-1))[0][0] for p, r in zip(p_emb, r_emb)]
        del st; torch.cuda.empty_cache()
        return float(np.mean(sims))

N_GEN = 300
gen_results = OrderedDict()
rouge = rouge_scorer.RougeScorer(['rouge1','rouge2','rougeL'], use_stemmer=True)

# Methods to evaluate
gen_methods = OrderedDict()
gen_methods['Oracle'] = None
gen_methods['Cosine'] = sel_cos
for mname, mdata in hp_models.items():
    gen_methods[mname] = make_sel(mdata['model'])

for gname, sel_fn in gen_methods.items():
    print(f'Generating for {gname}...')
    preds, refs, rscores = [], [], defaultdict(list)
    for raw, enc_s in tqdm(list(zip(hp_test, hp_te))[:N_GEN], desc=gname):
        if gname == 'Oracle':
            ctxs = [c['text'] for c in raw['contexts'] if c['is_supporting']]
        else:
            sel, _ = sel_fn(enc_s['query_emb'], enc_s['context_embs'], k=2)
            ctxs = [raw['contexts'][i]['text'] for i in sel]
        pred = gen_answer(raw['query'], ctxs)
        preds.append(pred); refs.append(raw['answer'])
        rs = rouge.score(raw['answer'], pred)
        for key in ['rouge1','rouge2','rougeL']: rscores[key].append(rs[key].fmeasure)
    bs = compute_bertscore(preds, refs)
    gen_results[gname] = {k: np.mean(v) for k,v in rscores.items()} | {'bertscore': bs}
    print(f'  ROUGE-L={gen_results[gname]["rougeL"]:.4f}, BERTScore={bs:.4f}')

print('\nGeneration quality summary:')
print(f'{"Method":15s} | {"ROUGE-L":>8s} | {"BERTSc":>8s} | {"vs Cos RL":>10s}')
cos_rl = gen_results['Cosine']['rougeL']
for gn, gr in gen_results.items():
    print(f'{gn:15s} | {gr["rougeL"]:8.4f} | {gr["bertscore"]:8.4f} | {gr["rougeL"]-cos_rl:+10.4f}')


---
## Part 7: Efficiency and Practical Analysis

In [ ]:
# ====== Latency and model size ======
test_sub = hp_te[:100]

def measure_lat(sf, data, k=2, runs=3):
    ts = []
    for _ in range(runs):
        t0 = time.perf_counter()
        for s in data: sf(s['query_emb'], s['context_embs'], k=k)
        ts.append((time.perf_counter()-t0)/len(data)*1000)
    return np.mean(ts), np.std(ts)

print('Efficiency analysis:')
print(f'{"Method":20s} | {"Latency(ms)":>12s} | {"Params":>12s} | {"Size(MB)":>10s} | {"vs Cosine":>10s}')
print('-'*75)

eff_data = OrderedDict()
cl, _ = measure_lat(sel_cos, test_sub)
eff_data['Cosine'] = {'lat': cl, 'params': 0, 'mb': 0}
print(f'{"Cosine":20s} | {cl:12.3f} | {0:>12d} | {0:10.1f} | {"1.0x":>10s}')

for mname, mdata in hp_models.items():
    lat, _ = measure_lat(make_sel(mdata['model']), test_sub)
    np_ = sum(p.numel() for p in mdata['model'].parameters())
    mb = sum(p.numel()*p.element_size() for p in mdata['model'].parameters())/1e6
    eff_data[mname] = {'lat': lat, 'params': np_, 'mb': mb}
    print(f'{mname:20s} | {lat:12.3f} | {np_:>12,} | {mb:10.1f} | {lat/cl:9.1f}x')

import matplotlib.pyplot as plt

# ====== Figure 1: Main Results (3 panels) ======

fig, axes = plt.subplots(1, 3, figsize=(9.0, 3.0))

# (a) F1 by geometry
order = ['Oracle','Random','Cosine'] + list(hp_models.keys())
vals = [results_hp[m]['f1'].mean() for m in order]
cols = [results_hp[m]['color'] for m in order]
labels_a = ['Oracle','Random','Cosine','Diag','LowRank','Skew','Mix\nK=2','Mix\nK=3']
bars = axes[0].bar(range(len(order)), vals, color=cols, alpha=0.9, edgecolor='#333', lw=0.5)
for b, v in zip(bars, vals):
    axes[0].text(b.get_x()+b.get_width()/2, v+0.008, f'{v:.3f}', ha='center', fontsize=5.5)
axes[0].set_xticks(range(len(order))); axes[0].set_xticklabels(labels_a, fontsize=6.5)
axes[0].set_ylabel('F1 (k=2)'); axes[0].set_title('(a) Retrieval F1 by geometry')
axes[0].set_ylim(0, 1.05)

# (b) Compression curve
ks = [1,2,3,4,5]
for rn in ['Cosine'] + list(hp_models.keys()):
    vals_k = [comp_hp[rn][k].mean() for k in ks]
    c = results_hp[rn]['color'] if rn in results_hp else '#333'
    ls = '--' if rn == 'Cosine' else '-'
    axes[1].plot(ks, vals_k, marker='o' if rn=='Cosine' else 's', ms=4, color=c, ls=ls, lw=1.2, label=rn)
axes[1].set_xlabel('k (selected)'); axes[1].set_ylabel('F1')
axes[1].set_title('(b) F1 vs compression level'); axes[1].legend(loc='upper center', bbox_to_anchor=(0.5, 1.35), ncol=4, fontsize=5.5)
axes[1].set_xticks(ks)

# (c) Generation ROUGE-L
gen_order = list(gen_results.keys())
gen_vals = [gen_results[m]['rougeL'] for m in gen_order]
gen_cols = ['#238B45','#969696','#6BAED6','#4292C6','#FD8D3C','#CB181D','#8C2D04'][:len(gen_order)]
gen_labs = ['Oracle','Cosine','Diag','LR','Skew','Mix2','Mix3'][:len(gen_order)]
bars_g = axes[2].bar(range(len(gen_order)), gen_vals, color=gen_cols, alpha=0.9, edgecolor='#333', lw=0.5)
for b, v in zip(bars_g, gen_vals):
    axes[2].text(b.get_x()+b.get_width()/2, v+0.005, f'{v:.3f}', ha='center', fontsize=5.5)
axes[2].set_xticks(range(len(gen_order))); axes[2].set_xticklabels(gen_labs, fontsize=6.5)
axes[2].set_ylabel('ROUGE-L'); axes[2].set_title('(c) Generation quality')

for ax in axes:
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.subplots_adjust(wspace=0.35, hspace=0.3)
plt.savefig('fig1_main_results.pdf', dpi=300)
plt.savefig('fig1_main_results.png', dpi=300)
plt.show(); print('Saved: fig1_main_results.pdf/.png')


In [ ]:
import matplotlib.pyplot as plt

# ====== Figure 1: Main Results (3 panels) with Optimized Layout ======
fig, axes = plt.subplots(1, 3, figsize=(12.0, 3.0))

# (a) F1 by geometry
order = ['Oracle','Random','Cosine'] + list(hp_models.keys())
vals = [results_hp[m]['f1'].mean() for m in order]
cols = [results_hp[m]['color'] for m in order]
labels_a = ['Oracle','Random','Cosine','Diag','LowRank','Skew','Mix\nK=2','Mix\nK=3']
bars = axes[0].bar(range(len(order)), vals, color=cols, alpha=0.9, edgecolor='#333', lw=0.5)
for b, v in zip(bars, vals):
    axes[0].text(b.get_x()+b.get_width()/2, v+0.008, f'{v:.3f}', ha='center', fontsize=5.5)
axes[0].set_xticks(range(len(order)))
axes[0].set_xticklabels(labels_a, fontsize=6.5, rotation=45, ha='right')
axes[0].set_ylabel('F1 (k=2)'); axes[0].set_title('(a) Retrieval F1 by geometry')
axes[0].set_ylim(0, 1.05)

# (b) Compression curve
ks = [1,2,3,4,5]
for rn in ['Cosine'] + list(hp_models.keys()):
    vals_k = [comp_hp[rn][k].mean() for k in ks]
    c = results_hp[rn]['color'] if rn in results_hp else '#333'
    ls = '--' if rn == 'Cosine' else '-'
    axes[1].plot(ks, vals_k, marker='o' if rn=='Cosine' else 's', ms=4, color=c, ls=ls, lw=1.2, label=rn)
axes[1].set_xlabel('k (selected)'); axes[1].set_ylabel('F1')
axes[1].set_title('(b) F1 vs compression level'); axes[1].legend(loc='upper center', bbox_to_anchor=(0.5, 1.35), ncol=4, fontsize=5.5)
axes[1].set_xticks(ks)

# (c) Generation ROUGE-L
gen_order = list(gen_results.keys())
gen_vals = [gen_results[m]['rougeL'] for m in gen_order]
gen_cols = ['#238B45','#969696','#6BAED6','#4292C6','#FD8D3C','#CB181D','#8C2D04'][:len(gen_order)]
gen_labs = ['Oracle','Cosine','Diag','LR','Skew','Mix2','Mix3'][:len(gen_order)]
bars_g = axes[2].bar(range(len(gen_order)), gen_vals, color=gen_cols, alpha=0.9, edgecolor='#333', lw=0.5)
for b, v in zip(bars_g, gen_vals):
    axes[2].text(b.get_x()+b.get_width()/2, v+0.005, f'{v:.3f}', ha='center', fontsize=5.5)
axes[2].set_xticks(range(len(gen_order)))
axes[2].set_xticklabels(gen_labs, fontsize=6.5, rotation=45, ha='right')
axes[2].set_ylabel('ROUGE-L'); axes[2].set_title('(c) Generation quality')

for ax in axes:
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.subplots_adjust(wspace=0.35, hspace=0.3, bottom=0.2)
plt.savefig('fig1_main_results.pdf', dpi=300)
plt.savefig('fig1_main_results.png', dpi=300)
plt.show()
import os
print(f"Files saved: {os.path.exists('fig1_main_results.pdf')} (pdf), {os.path.exists('fig1_main_results.png')} (png)")

In [ ]:
# ====== Figure 2: Ablation + Question Type (2 panels) ======

fig, axes = plt.subplots(1, 2, figsize=(7.0, 2.8))

# (a) Question type improvement
cos_by_type = {}
for t in types:
    idx = type_idx[t]
    cos_by_type[t] = np.mean([results_hp['Cosine']['f1'][i] for i in idx])

# Best model for each type
best_model_name = max([n for n in hp_models], key=lambda n: results_hp[n]['f1'].mean())
deltas_type = []
for t in types:
    idx = type_idx[t]
    our = np.mean([results_hp[best_model_name]['f1'][i] for i in idx])
    deltas_type.append(our - cos_by_type[t])

colors_dt = ['#238B45' if d>0 else '#CB181D' for d in deltas_type]
bars_t = axes[0].barh(range(len(types)), deltas_type, color=colors_dt, alpha=0.8, edgecolor='#333', lw=0.5)
axes[0].set_yticks(range(len(types)))
axes[0].set_yticklabels([f'{t} (n={len(type_idx[t])})' for t in types], fontsize=8)
axes[0].axvline(x=0, color='#333', lw=0.8)
axes[0].set_xlabel('F1 difference (best - cosine)')
axes[0].set_title(f'(a) Improvement by type ({best_model_name})')
for b, d in zip(bars_t, deltas_type):
    axes[0].text(d + (0.001 if d>=0 else -0.001), b.get_y()+b.get_height()/2,
                 f'{d:+.3f}', va='center', ha='left' if d>=0 else 'right', fontsize=7)

# (b) Training curves (val F1)
for mname, mdata in hp_models.items():
    c = model_colors[mname]
    axes[1].plot(mdata['hist']['val_f1'], color=c, lw=1.2, label=mname)
axes[1].axhline(y=cos_m, color='#969696', ls='--', lw=1, label='Cosine')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Validation F1')
axes[1].set_title('(b) Training dynamics')
axes[1].legend(fontsize=6, ncol=2)

for ax in axes:
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('fig2_ablation_type.pdf', dpi=300)
plt.savefig('fig2_ablation_type.png', dpi=300)
plt.show(); print('Saved: fig2_ablation_type.pdf/.png')

In [ ]:
# ====== Figure 3: Efficiency + Cross-dataset (2 panels) ======

fig, axes = plt.subplots(1, 2, figsize=(7.0, 2.8))

# (a) Efficiency-quality scatter
for mname, ed in eff_data.items():
    f1m = results_hp.get(mname, {}).get('f1', np.array([cos_m])).mean()
    c = results_hp.get(mname, {}).get('color', '#333')
    mk = 'o' if mname == 'Cosine' else 's'
    axes[0].scatter(ed['lat'], f1m, color=c, marker=mk, s=60, zorder=3, edgecolors='#333', lw=0.5)
    label = mname if len(mname) <= 10 else mname[:8]+'.'
    axes[0].annotate(label, (ed['lat'], f1m), textcoords='offset points', xytext=(6,-4), fontsize=6)
axes[0].set_xlabel('Latency (ms/sample)'); axes[0].set_ylabel('F1 (k=2)')
axes[0].set_title('(a) Efficiency-quality tradeoff')

# (b) Cross-dataset
ds_names = ['HotpotQA', 'SQuAD']
cos_vals = [results_hp['Cosine']['f1'].mean(), sq_cos.mean()]
our_vals = [results_hp.get('Skew', results_hp.get('LowRank', {})).get('f1', np.array([0])).mean(), sq_our.mean()]
x = np.arange(len(ds_names))
w = 0.35
b1 = axes[1].bar(x-w/2, cos_vals, w, color='#969696', alpha=0.9, label='Cosine', edgecolor='#333', lw=0.5)
b2 = axes[1].bar(x+w/2, our_vals, w, color='#FD8D3C', alpha=0.9, label='Skew (Ours)', edgecolor='#333', lw=0.5)
for b, v in zip(b1, cos_vals): axes[1].text(b.get_x()+b.get_width()/2, v+0.005, f'{v:.3f}', ha='center', fontsize=6)
for b, v in zip(b2, our_vals): axes[1].text(b.get_x()+b.get_width()/2, v+0.005, f'{v:.3f}', ha='center', fontsize=6)
axes[1].set_xticks(x); axes[1].set_xticklabels(ds_names)
axes[1].set_ylabel('F1'); axes[1].set_title('(b) Cross-dataset generalization')
axes[1].legend(fontsize=7)

for ax in axes:
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('fig3_efficiency_cross.pdf', dpi=300)
plt.savefig('fig3_efficiency_cross.png', dpi=300)
plt.show(); print('Saved: fig3_efficiency_cross.pdf/.png')

---
## Part 9: Final Summary

In [ ]:
print('#'*70)
print('#  Phase 2 Comprehensive Evaluation - Final Summary')
print('#'*70)

cos_m = results_hp['Cosine']['f1'].mean()
best_name = max([n for n in hp_models], key=lambda n: results_hp[n]['f1'].mean())
best_f1 = results_hp[best_name]['f1'].mean()

print(f'\n## Training Scale')
print(f'  HotpotQA training samples: {len(hp_tr)}')
print(f'  Test samples: {len(hp_te)}')

print(f'\n## Retrieval F1 (k=2, HotpotQA)')
for rn, rd in results_hp.items():
    fm = rd['f1'].mean(); d = fm - cos_m
    mk = '*' if d > 0 and rn not in ('Oracle','Random','Cosine') else ' '
    print(f'  {mk}{rn:15s}: F1={fm:.4f} (delta={d:+.4f})')

print(f'\n## Best model: {best_name} (F1={best_f1:.4f}, +{(best_f1-cos_m)*100:.2f}%)')

print(f'\n## Generation Quality (ROUGE-L)')
for gn, gr in gen_results.items():
    print(f'  {gn:15s}: RL={gr["rougeL"]:.4f} BS={gr["bertscore"]:.4f}')

print(f'\n## Cross-Dataset')
print(f'  HotpotQA Skew: {results_hp.get("Skew", {}).get("f1", np.array([0])).mean():.4f}')
print(f'  SQuAD Skew: {sq_our.mean():.4f} (Cosine: {sq_cos.mean():.4f})')

print(f'\n## Efficiency (best model: {best_name})')
ed = eff_data.get(best_name, {})
print(f'  Latency: {ed.get("lat",0):.2f} ms/sample')
print(f'  Parameters: {ed.get("params",0):,}')
print(f'  Model size: {ed.get("mb",0):.1f} MB')

print(f'\n## Geometric Form Ranking')
print(f'  Cosine (point) -> Diagonal -> LowRank -> Skew (best F1) -> Mixture (best ROUGE-L)')
print(f'  Each step adds geometric expressiveness and improves performance.')

print('\n' + '#'*70)

In [ ]:
# ====== Save everything ======
save = {
    'hp_results': {n: float(r['f1'].mean()) for n, r in results_hp.items()},
    'gen_results': gen_results,
    'comp_hp': {rn: {k: float(v.mean()) for k,v in cd.items()} for rn, cd in comp_hp.items()},
    'squad': {'cos': float(sq_cos.mean()), 'skew': float(sq_our.mean()), 'p': float(sq_pv)},
    'efficiency': {n: d for n, d in eff_data.items()},
    'config': {'train_size': len(hp_tr), 'test_size': len(hp_te), 'rank': RANK, 'temp': TEMP},
}
# Save best model
save['best_model_state'] = hp_models[best_name]['model'].state_dict()
save['best_model_name'] = best_name
torch.save(save, 'phase2_results.pt')
print('Saved: phase2_results.pt')